# MODEL D:  (DeBERTa v3)

**dominating Moral Value & Powerlessness**.


- **Model**: DeBERTa-v3-base (superior disentangled attention for nuanced rhetoric)
- **Loss**: Focal Loss with class-specific gamma (3.0 for weak classes)
- **Data**: 6x upsampling for weak classes, 0.3x for Economic
- **Context**: SHORT (128 tokens) - forces focus on local rhetoric cues
- **Cues**: Lexicon-based markers ([CUE_PWR], [CUE_MOR], [CUE_CON])
- **Training**: Freeze-then-unfreeze (300 steps), f1_weak_avg early stopping
- **Focus**: Moral Value & Powerlessness at all costs

**Class-Specific Configuration**:
```
                 Gamma  Alpha  Upsample  Focus
Economic         0.5    0.70   0.3x      SACRIFICE!
Conflict         1.0    1.20   1.0x      Low
Human Impact     1.0    1.00   1.0x      Low
None             1.0    1.00   1.0x      Low
Moral Value      3.0    2.00   6.0x      MAX 
Powerlessness    3.0    2.00   6.0x      MAX
```


**Role in Ensemble**: Provides strong weak-class predictions; ensemble uses 50% weight for Moral Value & Powerlessness.

In [5]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch.optim import AdamW

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Imports loaded")
print(f"Device: {device}")

Imports loaded
Device: cuda


## 1) Load Data with Aggressive Weak-Class Upsampling

In [6]:
print("Loading augmented data...")
augmented = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])
real_train = augmented[augmented['split'] == 'train'].copy()
real_val = augmented[augmented['split'] == 'validation'].copy()
real_test = augmented[augmented['split'] == 'test'].copy()
print(f"Real train: {len(real_train)}")

print("\nLoading pseudo-labels (confidence >= 0.90)...")
pseudo = pd.read_csv('data/pseudo_labeled_chunks.csv', keep_default_na=False, na_values=[''])
pseudo = pseudo[pseudo['confidence'] >= 0.90].copy()
pseudo.rename(columns={'pseudo_label': 'frame_label'}, inplace=True)

# AGGRESSIVE upsampling: 6x for weak classes, 0.3x for Economic
target_labels = ['Moral Value', 'Powerlessness']
pseudo['sample_weight'] = pseudo['frame_label'].apply(
    lambda x: 0.8 if x in target_labels else 0.2
)
real_train['sample_weight'] = real_train['frame_label'].apply(
    lambda x: 6.0 if x in target_labels else (0.3 if x == 'Economic' else 1.0)
)

combined_train = pd.concat([real_train, pseudo], ignore_index=True)
print(f"\nCombined: {len(combined_train)}")
print(f"Pseudo/Real ratio: {len(pseudo)/len(real_train):.2f}:1")

print("\nEffective upsampling per class:")
for label in combined_train['frame_label'].unique():
    subset = combined_train[combined_train['frame_label'] == label]
    weighted = subset['sample_weight'].sum()
    print(f"  {label:20s}: {len(subset):4d} samples × {subset['sample_weight'].mean():.1f} = {weighted:.0f} effective")

Loading augmented data...


Real train: 4606

Loading pseudo-labels (confidence >= 0.90)...

Combined: 13606
Pseudo/Real ratio: 1.95:1

Effective upsampling per class:
  Human Impact        : 2022 samples × 0.4 = 822 effective
  Conflict            : 2431 samples × 0.5 = 1231 effective
  Economic            : 2146 samples × 0.2 = 494 effective
  Powerlessness       : 2432 samples × 2.8 = 6792 effective
  Moral Value         : 2442 samples × 2.8 = 6852 effective
  None                : 2133 samples × 0.4 = 933 effective


## 2) Lexicon-Based Cue Injection

In [7]:
# Define rhetoric cue lexicons
pwr_cues = ['powerless', 'cannot', 'cant', 'forced', 'helpless', 'against them', 
            'unfair', 'abused', 'victim', 'no choice', 'no say', 'trapped']
mor_cues = ['immoral', 'unjust', 'wrong', 'corrupt', 'violate', 'violation', 
            'ethical', 'unethical', 'morality', 'values', 'principle', 'duty']
con_cues = ['conflict', 'fight', 'attack', 'versus', 'vs', 'threat', 'enemy', 
            'clash', 'violence', 'battle', 'war', 'oppose']

def add_cue_markers(text: str) -> str:
    """Prepend rhetoric cue markers based on lexicon matches."""
    t = text.lower()
    prefixes = []
    
    if any(c in t for c in pwr_cues):
        prefixes.append('[CUE_PWR]')
    if any(c in t for c in mor_cues):
        prefixes.append('[CUE_MOR]')
    if any(c in t for c in con_cues):
        prefixes.append('[CUE_CON]')
    
    return (' '.join(prefixes) + ' ' + text) if prefixes else text

# Apply to all datasets
combined_train['chunk_text_with_cues'] = combined_train['chunk_text'].astype(str).apply(add_cue_markers)
real_val['chunk_text_with_cues'] = real_val['chunk_text'].astype(str).apply(add_cue_markers)
real_test['chunk_text_with_cues'] = real_test['chunk_text'].astype(str).apply(add_cue_markers)

# Show examples
print("Examples with cues:")
for i in range(3):
    sample = combined_train.iloc[i]
    if '[CUE' in sample['chunk_text_with_cues']:
        print(f"\n{sample['frame_label']:20s}: {sample['chunk_text_with_cues'][:150]}...")
        break

print("\n✅ Cue markers added")

Examples with cues:

Human Impact        : [CUE_MOR] Like my good friend the right hon. Member for Normanton, Pontefract and Castleford (Yvette Cooper), I shall be voting against this Bill. It ...

✅ Cue markers added


## 3) Encode & Tokenize (SHORT CONTEXT = 128)

In [8]:
with open('data/roberta_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
labels = list(label_encoder.classes_)
num_labels = len(labels)

combined_train['label'] = label_encoder.transform(combined_train['frame_label'])
real_val['label'] = label_encoder.transform(real_val['frame_label'])
real_test['label'] = label_encoder.transform(real_test['frame_label'])

# DeBERTa tokenizer with SHORT context (128 tokens)
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
max_length = 358  # Longer context for better semantic understanding

def tokenize_function(examples):
    return tokenizer(
        examples['chunk_text_with_cues'],
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )

train_dataset = Dataset.from_pandas(combined_train[['chunk_text_with_cues', 'label', 'sample_weight']])
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text_with_cues'])
train_dataset.set_format('torch')

val_dataset = Dataset.from_pandas(real_val[['chunk_text_with_cues', 'label']])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text_with_cues'])
val_dataset.set_format('torch')

test_dataset = Dataset.from_pandas(real_test[['chunk_text_with_cues', 'label']])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text_with_cues'])
test_dataset.set_format('torch')

print(f"Datasets: {len(train_dataset)}, {len(val_dataset)}, {len(test_dataset)}")
print(f"Max length: {max_length} tokens (SHORT context)")

Map:   0%|          | 0/13606 [00:00<?, ? examples/s]

Map:   0%|          | 0/683 [00:00<?, ? examples/s]

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

Datasets: 13606, 683, 348
Max length: 358 tokens (SHORT context)


## 4) Focal Loss Trainer with Class-Specific Gamma

In [9]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorWithSampleWeight(DataCollatorWithPadding):
    """Custom collator to preserve sample_weight."""
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        sample_weights = None
        if 'sample_weight' in features[0]:
            sample_weights = torch.FloatTensor([f.pop('sample_weight') for f in features])
        batch = super().__call__(features)
        if sample_weights is not None:
            batch['sample_weight'] = sample_weights
        return batch

class FocalLoss(nn.Module):
    """Focal Loss with per-class gamma and alpha."""
    def __init__(self, gamma_per_class, alpha=None):
        super().__init__()
        self.gamma = torch.tensor(gamma_per_class)
        self.alpha = torch.tensor(alpha) if alpha is not None else None
    
    def forward(self, logits, targets):
        ce = nn.CrossEntropyLoss(reduction='none')(logits, targets)
        pt = torch.exp(-ce)
        
        # Class-specific gamma
        gamma = self.gamma.to(logits.device)[targets]
        focal_term = ((1 - pt) ** gamma) * ce
        
        # Class-specific alpha (weights)
        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)[targets]
            focal_term = focal_term * alpha
        
        return focal_term

# AGGRESSIVE gamma for weak classes (3.0), low for Economic (0.5)
gamma_per_class = [1.0, 0.5, 1.0, 3.0, 1.0, 3.0]  # [Conflict, Economic, Human, Moral, None, Power]
alpha_weights = [1.2, 0.7, 1.0, 2.0, 1.0, 2.0]

focal_loss = FocalLoss(gamma_per_class=gamma_per_class, alpha=alpha_weights)

class FocalTrainer(Trainer):
    """Trainer with Focal Loss."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        sample_weight = inputs.pop('sample_weight', None)
        
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss_vec = focal_loss(logits, labels)
        
        if sample_weight is not None:
            loss = (loss_vec * sample_weight).mean()
        else:
            loss = loss_vec.mean()
        
        return (loss, outputs) if return_outputs else loss

print("Focal Loss Trainer defined")
print(f"Gamma per class: {gamma_per_class}")
print(f"Alpha weights: {alpha_weights}")

Focal Loss Trainer defined
Gamma per class: [1.0, 0.5, 1.0, 3.0, 1.0, 3.0]
Alpha weights: [1.2, 0.7, 1.0, 2.0, 1.0, 2.0]


## 5) Load DeBERTa Model (Encoder Frozen Initially)

In [10]:
print("Loading DeBERTa-v3-base...")
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base',
    num_labels=num_labels,
    attention_probs_dropout_prob=0.10,
    hidden_dropout_prob=0.10,
    ignore_mismatched_sizes=True,
    use_safetensors=True  # Force safetensors to avoid torch.load security issue
)
model = model.to(device)

# FREEZE encoder initially (unfreeze at step 300)
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

print("Model loaded with safetensors")
print("Encoder: FROZEN (will unfreeze at step 300)")
print("Classifier: TRAINABLE")

Loading DeBERTa-v3-base...


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with safetensors
Encoder: FROZEN (will unfreeze at step 300)
Classifier: TRAINABLE


## 6) Freeze–Unfreeze Callback

In [11]:
class UnfreezeEncoderCallback(EarlyStoppingCallback):
    """Unfreezes encoder at specified step, then applies early stopping."""
    def __init__(self, unfreeze_at_steps=300, early_stopping_patience=5):
        super().__init__(early_stopping_patience=early_stopping_patience)
        self.unfreeze_at_steps = unfreeze_at_steps
        self.done = False
    
    def on_step_end(self, args, state, control, **kwargs):
        if not self.done and state.global_step >= self.unfreeze_at_steps:
            for name, param in model.named_parameters():
                if 'classifier' not in name:
                    param.requires_grad = True
            self.done = True
            print(f"\n🔓 Encoder unfrozen at step {state.global_step}")
        return control

print("Freeze–Unfreeze callback defined")
print("  Encoder unfreezes at step 300")
print("  Early stopping patience: 5")

Freeze–Unfreeze callback defined
  Encoder unfreezes at step 300
  Early stopping patience: 5


## 7) Optimizer with Differential LR

In [12]:
encoder_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]

encoder_lr = 1e-5       # VERY LOW (fine-tune after unfreeze)
classifier_lr = 1.5e-3  # MODERATE (aggressive learning on weak classes)

optimizer = AdamW([
    {'params': encoder_params, 'lr': encoder_lr},
    {'params': classifier_params, 'lr': classifier_lr}
], weight_decay=0.05)

# Cosine schedule
num_training_steps = len(train_dataset) // 8 // 4 * 12
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f"Encoder LR: {encoder_lr}")
print(f"Classifier LR: {classifier_lr}")
print(f"Training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

Encoder LR: 1e-05
Classifier LR: 0.0015
Training steps: 5100
Warmup steps: 510


## 8) Compute Metrics (f1_weak_avg for early stopping)

In [13]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average='macro')
    f1_weighted = f1_score(labels_np, preds, average='weighted')
    
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels_np, preds, average=None, zero_division=0
    )
    
    # Custom metric: average F1 of weak classes (Moral Value + Powerlessness)
    f1_weak_avg = (f1_per_class[3] + f1_per_class[5]) / 2
    
    metrics = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'f1_weak_avg': float(f1_weak_avg)  # KEY METRIC for early stopping
    }
    
    for i, label in enumerate(labels):
        metrics[f'f1_{label}'] = f1_per_class[i]
    
    return metrics

print("Metrics defined")
print("Early stopping based on: f1_weak_avg (Moral Value + Powerlessness)")

Metrics defined
Early stopping based on: f1_weak_avg (Moral Value + Powerlessness)


## 9) Training (12 Epochs, Patience=5)

In [14]:
training_args = TrainingArguments(
    output_dir='models/deberta-ensemble-model-D',
    num_train_epochs=12,
    per_device_train_batch_size=4,  # Reduced from 8 for 6GB GPU
    per_device_eval_batch_size=16,  # Reduced from 32 for 6GB GPU
    gradient_accumulation_steps=4,  # Maintains effective batch size of 16
    weight_decay=0.05,
    max_grad_norm=2.0,  # Tight clipping for stability
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_weak_avg',  # CRITICAL: optimize for weak classes
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to='none',
    seed=42,
    torch_empty_cache_steps=50,  # Clear cache every 50 steps
    gradient_checkpointing=False,  # Disabled: conflicts with custom Focal Loss
)

data_collator = DataCollatorWithSampleWeight(tokenizer=tokenizer, padding=True)

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[UnfreezeEncoderCallback(unfreeze_at_steps=300, early_stopping_patience=5)],
    data_collator=data_collator,
    optimizers=(optimizer, scheduler)
)

print("=" * 80)
print("MODEL D: RHETORIC SPECIALIST (DeBERTa v3)")
print("=" * 80)
print("Strategy: Dominate Moral Value & Powerlessness")
print("Focal Loss: gamma=[1.0, 0.5, 1.0, 3.0, 1.0, 3.0]")
print("Context: 128 tokens (SHORT)")
print("Upsampling: 6x weak, 0.3x Economic")
print("Freeze-Unfreeze: Step 300")
print("Early Stop: f1_weak_avg (patience=5)")
print("Target: Moral Value & Powerlessness F1 > 0.68")
print("=" * 80)

train_result = trainer.train()
print("\n✅ Training complete!")

MODEL D: RHETORIC SPECIALIST (DeBERTa v3)
Strategy: Dominate Moral Value & Powerlessness
Focal Loss: gamma=[1.0, 0.5, 1.0, 3.0, 1.0, 3.0]
Context: 128 tokens (SHORT)
Upsampling: 6x weak, 0.3x Economic
Freeze-Unfreeze: Step 300
Early Stop: f1_weak_avg (patience=5)
Target: Moral Value & Powerlessness F1 > 0.68


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Weak Avg,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
50,1.659900,1.600062,0.166911,0.047799,0.047869,0.143396,0.000000,0.000000,0.000000,0.286792,0.000000,0.000000
100,1.578500,1.575933,0.166911,0.047679,0.047749,0.143036,0.000000,0.000000,0.000000,0.286073,0.000000,0.000000
150,1.543700,1.591199,0.153734,0.062038,0.061986,0.186115,0.000000,0.000000,0.000000,0.274453,0.000000,0.097778
200,1.570500,1.582539,0.165447,0.047320,0.046974,0.141960,0.000000,0.000000,0.000000,0.000000,0.000000,0.283920
250,1.563100,1.580548,0.166911,0.047679,0.047749,0.143036,0.000000,0.000000,0.000000,0.286073,0.000000,0.000000
300,1.569000,1.600514,0.178624,0.086863,0.086573,0.260589,0.000000,0.000000,0.000000,0.236422,0.000000,0.284757
350,1.562000,1.559776,0.168375,0.055682,0.055703,0.167047,0.000000,0.000000,0.000000,0.292428,0.000000,0.041667
400,1.434300,1.413081,0.226940,0.144866,0.137817,0.142462,0.000000,0.000000,0.000000,0.000000,0.584270,0.284924
450,1.042800,1.217965,0.430454,0.372018,0.366965,0.250000,0.515021,0.000000,0.570621,0.136364,0.646465,0.363636
500,0.795500,1.118741,0.472914,0.445850,0.442162,0.400625,0.524476,0.135338,0.582456,0.465116,0.631579,0.336134



🔓 Encoder unfrozen at step 300

✅ Training complete!


## 10) Evaluate & Save

In [15]:
print("\nValidation...")
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(val_results)

print("\nTest...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(test_results)

# Save model
os.makedirs('models/deberta-ensemble-model-D', exist_ok=True)
trainer.save_model('models/deberta-ensemble-model-D')
tokenizer.save_pretrained('models/deberta-ensemble-model-D')

# Save metrics
os.makedirs('results', exist_ok=True)
with open('results/model_D_metrics.json', 'w') as f:
    json.dump({
        'model': 'D - Rhetoric Specialist (DeBERTa v3)',
        'strategy': 'Focal loss + cues + freeze-unfreeze + short context',
        'gamma_per_class': gamma_per_class,
        'alpha_weights': alpha_weights,
        'max_length': max_length,
        'val': val_results,
        'test': test_results,
        'time': train_result.metrics['train_runtime']
    }, f, indent=2)

print("\n" + "="*80)
print("MODEL D RESULTS:")
print(f"Val F1: {val_results['eval_f1_macro']:.4f}")
print(f"Val F1 (weak avg): {val_results['eval_f1_weak_avg']:.4f}")
print(f"Test F1: {test_results['eval_f1_macro']:.4f}")
print(f"Test F1 (weak avg): {test_results['eval_f1_weak_avg']:.4f}")
print("\nPer-Class F1:")
for label in labels:
    print(f"  {label:20s}: {test_results[f'eval_f1_{label}']:.4f}")
print("="*80)


Validation...


{'eval_loss': 0.9592751860618591, 'eval_accuracy': 0.6691068814055637, 'eval_f1_macro': 0.6709597510569626, 'eval_f1_weighted': 0.6723888929074101, 'eval_f1_weak_avg': 0.5956778309409889, 'eval_f1_Conflict': 0.6186440677966102, 'eval_f1_Economic': 0.775330396475771, 'eval_f1_Human Impact': 0.7777777777777778, 'eval_f1_Moral Value': 0.5877192982456141, 'eval_f1_None': 0.6626506024096386, 'eval_f1_Powerlessness': 0.6036363636363636, 'eval_runtime': 33.6363, 'eval_samples_per_second': 20.305, 'eval_steps_per_second': 1.278, 'epoch': 1.4103468547912992}

Test...
{'eval_loss': 0.8345633149147034, 'eval_accuracy': 0.6379310344827587, 'eval_f1_macro': 0.6420324113937587, 'eval_f1_weighted': 0.6420324113937588, 'eval_f1_weak_avg': 0.5757763975155279, 'eval_f1_Conflict': 0.6355140186915887, 'eval_f1_Economic': 0.6796116504854369, 'eval_f1_Human Impact': 0.7563025210084033, 'eval_f1_Moral Value': 0.6086956521739131, 'eval_f1_None': 0.6292134831460674, 'eval_f1_Powerlessness': 0.5428571428571428,